# Setup Slice with perfSONAR and Custom Result Archiver Stack

This notebook demonstrates how to deploy and operate a **perfSONAR-based network measurement setup** across a **ship-side (edge)** and **shore-side (core)** environment within the **FABRIC Testbed**.

The configuration enables **automated, bidirectional performance testing** and **result archiving** on both ends — providing a resilient, self-contained monitoring framework suitable for distributed or mobile network scenarios such as research vessels or field stations.


## System Architecture

The experiment involves two VMs connected via **multiple FabNetv4 Network Services**:

![Architecture](./images/perfsonar-deployment.png)

### **Ship VM**

* Has **multiple NICs**, each on a different **FabNetv4 L3 network** (e.g., net1, net2)
* Runs **perfSONAR Testpoint** (Docker-based) that measures each network path independently using `--source` binding
* Runs a **custom lightweight pScheduler Result Archiver stack**, including:

  * **TimescaleDB** (time-series database)
  * **Grafana** – accessible at `https://127.0.0.1:8443` (via SSH tunnel)
  * **Archiver**
  * **Nginx**
* Archives results both **locally** and to the **remote Archiver** instance on the Shore VM.



### **Shore VM**

* Has **multiple NICs**, one per FabNetv4 network, matching the ship's network topology
* Runs **perfSONAR Testpoint**
* Runs a **custom lightweight pScheduler Result Archiver stack**, including:

  * **TimescaleDB**
  * **Grafana** – accessible at `https://127.0.0.1:8443` (via SSH tunnel)
  * **Archiver**
  * **Nginx**
* Executes periodic **latency, RTT, MTU, clock, throughput,** and **trace tests every 6 hours**
* Archives results locally and on the Ship VM's Archiver instance for redundancy.


### **Multi-Network Measurement**

When the ship VM has multiple NICs on different L3 networks, the HOSTS specification uses `%` as a "via" separator to bind each test to a specific source:

```
dst_ip@name%src_ip       # bind to a specific source IP
dst_ip@name%iface_name   # resolve interface IP at runtime (survives DHCP renewals)
```

This ensures pScheduler tests are bound to the correct egress interface, enabling independent measurement of each network path. The source can be specified as an IP address or a network interface name — when an interface name is given, its current IP is resolved at runtime, making the configuration resilient to IP changes.


## Motivation for the Custom Archiver Stack

The **standard perfSONAR Toolkit** (including *perfsonar-grafana-toolkit* and *perfsonar-archive*) has a **high memory footprint**, making it unsuitable for **edge devices** such as **Raspberry Pis** typically deployed on smaller vessels or remote sites.

To address this, a **customized pScheduler Result Archiver stack** was developed:

* Streamlined to minimize memory and CPU usage.
* Modularized for independent deployment of TimescaleDB, Grafana, Archiver, and Nginx containers.
* Optimized for operation on low-resource ARM-based devices while maintaining full compatibility with the standard perfSONAR ecosystem.

This allows for **continuous and autonomous performance monitoring** even on constrained edge hardware, while still integrating seamlessly with shore-based or cloud-based analysis systems.

### 1. Import the FABlib Library

FABlib is used to programmatically create and manage FABRIC resources such as slices and VMs.

In [2]:
# ------------------------------------------------------------
# Step 1: Initialize FABlib
# ------------------------------------------------------------
# FABlib is used to create, configure, and manage the FABRIC
# resources required for this perfSONAR experiment.

from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress, os, json

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
# fablib.show_config();

### 2. Create the Experiment Slice

This step provisions the virtual resources (shore-side and ship-side VMs) on the FABRIC testbed.

In [3]:
# ------------------------------------------------------------
# Step 2: Define the experiment topology
# ------------------------------------------------------------
# This experiment uses two fixed FABRIC sites for reproducibility:
#
#   Ship testpoint  : TACC
#   Shore testpoint : NCSA
#
# Two independent FabNetv4 paths are created between the
# same endpoints so that their network performance can be compared.

slice_name = 'AperfSonar-fabric'
#sites = fablib.get_random_sites(count=3)
# [site1,site2] = sites = ["STAR", "FIU"]
[site1,site2] = sites = ["NCSA", "TACC"]
print(f"Sites: {sites}")

shore_name = "shore-testpoint"
# ship_name = "ship-testpoint"
ship_name = "ship-testpoint-new"
# Network names for multi-NIC setup (2 independent FabNetv4 networks)
net1_name = "net1"
net2_name = "net2"

Sites: ['NCSA', 'TACC']


In [9]:
# ------------------------------------------------------------
# Step 3: Create the FABRIC slice
# ------------------------------------------------------------
# Create the ship and shore virtual machines and connect them
# using two independent FabNetv4 Layer-3 networks.
#
# The two network paths allow perfSONAR to measure the same
# destination through different source interfaces.

slice = fablib.new_slice(name=slice_name)

# Shore node with 2 NICs (one per network)
shore = slice.add_node(name=shore_name, site=site1, image="default_ubuntu_24", cores=8, ram=16, disk=100)
shore.add_fabnet()

# Ship node with 2 NICs (one per network)
ship = slice.add_node(name=ship_name, site=site2, image="default_ubuntu_24", cores=4, ram=16, disk=100)
ship.add_fabnet(name=net1_name)
ship.add_fabnet(name=net2_name)

#Submit Slice Request
slice.submit();




Retry: 10, Time: 246 sec


ID,2d7fbe90-b4b6-453f-9dc9-e397eb271588
Name,AperfSonar-fabric
Lease Expiration (UTC),2026-09-05 19:23:41 +0000
Lease Start (UTC),2026-09-04 19:23:41 +0000
Project ID,a70de2f5-9e12-4b6b-b412-0ae1a2c553b0
State,StableOK
Email,ranna01@jaguar.tamu.edu
UserId,e335f96a-332c-49ec-9810-0d6cbe323c83


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
999a7c1a-2da3-4f82-a4f3-cdfbb4e57516,ship-testpoint,4,16,100,default_ubuntu_24,qcow2,tacc-w4.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fed2:64c2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fed2:64c2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
e2f5c1e4-5b17-41f0-ae35-bace8c729bc5,shore-testpoint,8,16,100,default_ubuntu_24,qcow2,ncsa-w2.fabric-testbed.net,NCSA,ubuntu,2620:0:c80:1001:f816:3eff:fe5b:7a0d,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:c80:1001:f816:3eff:fe5b:7a0d,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
c2ab30d7-6694-4982-8b74-ea812ea21e2f,FABNET_IPv4_NCSA,L3,FABNetv4,NCSA,10.132.137.1,10.132.137.0/24,Active,
4c8905bd-992e-48ca-97c7-7232fbf9531c,net1_IPv4_TACC,L3,FABNetv4,TACC,10.130.151.1,10.130.151.0/24,Active,
7fbedb5a-b291-4333-9f91-1a4f05674c81,net2_IPv4_TACC,L3,FABNetv4,TACC,10.130.150.1,10.130.150.0/24,Active,


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
shore-testpoint-FABNET_IPv4_NCSA_nic-p1,p1,shore-testpoint,FABNET_IPv4_NCSA,100,,12:53:E3:FD:D0:2E,enp7s0,enp7s0,auto,10.132.137.2,4,HundredGigE0/0/0/9
ship-testpoint-net2_IPv4_TACC_nic-p1,p1,ship-testpoint,net2_IPv4_TACC,100,,1E:89:33:CB:74:70,enp8s0,enp8s0,auto,10.130.150.2,4,HundredGigE0/0/0/11
ship-testpoint-net1_IPv4_TACC_nic-p1,p1,ship-testpoint,net1_IPv4_TACC,100,,1E:84:21:A5:BD:99,enp7s0,enp7s0,auto,10.130.151.2,4,HundredGigE0/0/0/11



Time to print interfaces 246 seconds



### 3. Configure Shore VM with perfSONAR Testpoint and Result Archiver Stack

* Deploy the **perfSONAR Testpoint** on the shore-side VM.
* Launch the **Result Archiver stack**, which includes:

  * **TimescaleDB** – for storing measurement results
  * **Grafana** – for data visualization and dashboards
  * **Archiver** – for collecting and managing test results


In [14]:
# ARCHIVER_TOKEN=!openssl rand -hex 32
# ARCHIVER_TOKEN = ARCHIVER_TOKEN[0]
import secrets

ARCHIVER_TOKEN = secrets.token_hex(32)
# Repository containing the setup scripts
RECIPES_REPO = "https://github.com/kthare10/fabric-recipes.git"
RECIPES_BRANCH = "main"
SCRIPTS_DIR = "fabric-recipes/perfsonar/node_tools"

# Branch overrides for archiver and perfsonar-extensions repos
# (setup_ship.sh and setup_shore.sh will clone these branches)
ARCHIVER_BRANCH = "main"
PSX_BRANCH = "main"

# NMEA Navigation Data Listener — set to True to deploy the nmea-listener
# container on the ship node. Only needed for vessels with NMEA 0183 GPS/MRU
# broadcasts (e.g., R/V Thompson). Set to False for non-vessel deployments.
# ENABLE_NMEA = True
ENABLE_NMEA = False
NMEA_PORT = 13551
VESSEL_ID = "ship-testpoint"

# import secrets

# ARCHIVER_TOKEN = secrets.token_hex(32)

# print("Archiver token generated successfully.")

# # Repository containing the setup scripts
# RECIPES_REPO = "https://github.com/kthare10/fabric-recipes.git"
# RECIPES_BRANCH = "main"
# SCRIPTS_DIR = "fabric-recipes/perfsonar/node_tools"

# # Branch overrides for archiver and perfsonar-extensions repos
# ARCHIVER_BRANCH = "main"
# PSX_BRANCH = "main"

# # NMEA disabled for this experiment
# ENABLE_NMEA = False
# NMEA_PORT = 13551
# VESSEL_ID = "ship-testpoint"

In [11]:
shore.execute(f'git clone --branch {RECIPES_BRANCH} --depth 1 {RECIPES_REPO} || git -C fabric-recipes pull')
stdout, stderr = shore.execute(f'{SCRIPTS_DIR}/setup_shore.sh --token {ARCHIVER_TOKEN} --archiver-branch {ARCHIVER_BRANCH}',
                              quiet=True, output_file=f"{shore.get_name()}.log")

fatal: destination path 'fabric-recipes' already exists and is not an empty directory.
Already up to date.


**This script should be executed twice. The initial run ensures Docker installation and group membership. The second run then proceeds with configuration, relying on the user's updated Docker permissions.**

### 4. Configure Ship VM with perfSONAR Testpoint and Local Archiver Stack

* Launch the **Result Archiver stack**, consisting of:

  * **TimescaleDB** – for local result storage
  * **Grafana** – for onboard visualization
  * **Archiver** – for local and remote result archiving
* Start the **perfSONAR Testpoint** container, configured to:

  * Automatically execute periodic network tests (default interval: every 6 hours)

In [15]:
ship.execute(f'git clone --branch {RECIPES_BRANCH} --depth 1 {RECIPES_REPO} || git -C fabric-recipes pull')

fatal: destination path 'fabric-recipes' already exists and is not an empty directory.
Already up to date.


('Already up to date.\n',
 "fatal: destination path 'fabric-recipes' already exists and is not an empty directory.\n")

In [29]:
nmea_flags = f' --nmea-port {NMEA_PORT} --vessel-id {VESSEL_ID}' if ENABLE_NMEA else ''

stdout, stderr = ship.execute(
    f'{SCRIPTS_DIR}/setup_ship.sh --token {ARCHIVER_TOKEN} --hosts "{hosts}" --urls {archive_urls}'
    f' --archiver-branch {ARCHIVER_BRANCH} --psx-branch {PSX_BRANCH}'
    f'{nmea_flags}',
    quiet=True, output_file=f"{ship.get_name()}.log")

In [12]:
# Get interface names for each network on the ship node (2 NICs)
ship_iface_net1 = ship.get_interface(network_name=f"{net1_name}_IPv4_{ship.get_site()}").get_os_interface()
ship_iface_net2 = ship.get_interface(network_name=f"{net2_name}_IPv4_{ship.get_site()}").get_os_interface()
print(f"Ship interfaces: net1={ship_iface_net1}  net2={ship_iface_net2}")

# Build HOSTS string with %iface syntax for multi-network measurement
# Ship measures to shore (net1) via two different source interfaces
# Format: dst_ip@name%iface  (interface IP resolved at runtime inside container)
host_via_net1 = f"{shore_addr_net1}@{shore.get_name()}-via-net1%{ship_iface_net1}"
host_via_net2 = f"{shore_addr_net1}@{shore.get_name()}-via-net2%{ship_iface_net2}"
hosts = f"{host_via_net1} {host_via_net2}"
print(f"Test Host Set: {hosts}")

# Use shore address for remote archiver endpoint
archive_urls = f"https://localhost:8443/ps,https://{shore_addr_net1}:8443/ps"
print(f"Archiver URLs Set: {archive_urls}")

Ship interfaces: net1=enp8s0  net2=enp7s0
Test Host Set: 10.132.137.2@shore-testpoint-via-net1%enp8s0 10.132.137.2@shore-testpoint-via-net2%enp7s0
Archiver URLs Set: https://localhost:8443/ps,https://10.132.137.2:8443/ps


/tmp/ipykernel_47/3354086529.py:2: DeprecationWarning: get_os_interface() is deprecated and will be removed in a future release, please use 'get_device_name()' instead.
  ship_iface_net1 = ship.get_interface(network_name=f"{net1_name}_IPv4_{ship.get_site()}").get_os_interface()
/tmp/ipykernel_47/3354086529.py:3: DeprecationWarning: get_os_interface() is deprecated and will be removed in a future release, please use 'get_device_name()' instead.
  ship_iface_net2 = ship.get_interface(network_name=f"{net2_name}_IPv4_{ship.get_site()}").get_os_interface()


**This script should be executed twice. The initial run ensures Docker installation and group membership. The second run then proceeds with configuration, relying on the user's updated Docker permissions.**

### 4a. Test Docker Deployment — Shore VM

Verify all Docker containers are running and healthy on the shore-side VM. The archiver stack should include: **timescaledb**, **grafana**, **archiver**, and **archiver-nginx**.

In [31]:
stdout, stderr = shore.execute(
    f'{SCRIPTS_DIR}/validate_deployment.sh --role shore --token {ARCHIVER_TOKEN}'
)

SHORE VM — Docker Deployment Validation

--- 1. Container Status ---
NAMES                    STATUS                 PORTS
archiver                 Up 5 weeks (healthy)   3500/tcp
grafana                  Up 5 weeks             3000/tcp
grafana-image-renderer   Up 5 weeks (healthy)   8081/tcp
archiver-nginx           Up 4 weeks             80/tcp, 0.0.0.0:8443->443/tcp, [::]:8443->443/tcp
timescaledb              Up 5 weeks (healthy)   5432/tcp
  [PASS] Container 'timescaledb' running


In [33]:
shore.execute(f'git -C fabric-recipes pull || true')
stdout, stderr = shore.execute(f'{SCRIPTS_DIR}/setup_shore.sh --token {ARCHIVER_TOKEN} --archiver-branch {ARCHIVER_BRANCH}',
                              quiet=True, output_file=f"{shore.get_name()}.log")

Already up to date.


### 4b. Test Docker Deployment — Ship VM

Verify all Docker containers are running on the ship-side VM. The ship runs the archiver stack **plus** the **perfsonar-testpoint** container.

In [34]:
stdout, stderr = ship.execute(
    f'{SCRIPTS_DIR}/validate_deployment.sh --role ship --token {ARCHIVER_TOKEN}'
)

print(stdout)
print(stderr)

SHIP VM — Docker Deployment Validation

--- 1. Container Status ---
NAMES                    STATUS                   PORTS
perfsonar-testpoint      Up 8 minutes (healthy)   
archiver                 Up 9 minutes (healthy)   3500/tcp
grafana                  Up 9 minutes             3000/tcp
grafana-image-renderer   Up 9 minutes (healthy)   8081/tcp
archiver-nginx           Up 9 minutes             80/tcp, 0.0.0.0:8443->443/tcp, [::]:8443->443/tcp
timescaledb              Up 9 minutes (healthy)   5432/tcp
  [PASS] Container 'timescaledb' running
SHIP VM — Docker Deployment Validation

--- 1. Container Status ---
NAMES                    STATUS                   PORTS
perfsonar-testpoint      Up 8 minutes (healthy)   
archiver                 Up 9 minutes (healthy)   3500/tcp
grafana                  Up 9 minutes             3000/tcp
grafana-image-renderer   Up 9 minutes (healthy)   8081/tcp
archiver-nginx           Up 9 minutes             80/tcp, 0.0.0.0:8443->443/tcp, [::]:8443->443/

In [18]:
nmea_flag = ' --nmea' if ENABLE_NMEA else ''

stdout, stderr = ship.execute(
    f'{SCRIPTS_DIR}/validate_deployment.sh --role ship --token {ARCHIVER_TOKEN}{nmea_flag}'
)

SHIP VM — Docker Deployment Validation

--- 1. Container Status ---
NAMES                    STATUS                PORTS
archiver                 Up 3 days (healthy)   3500/tcp
grafana                  Up 3 days             3000/tcp
grafana-image-renderer   Up 3 days (healthy)   8081/tcp
perfsonar-testpoint      Up 3 days (healthy)   
archiver-nginx           Up 3 days             80/tcp, 0.0.0.0:8443->443/tcp, [::]:8443->443/tcp
timescaledb              Up 3 days (healthy)   5432/tcp
  [PASS] Container 'timescaledb' running


In [68]:
stdout, stderr = shore.execute(
    "cd /home/ubuntu/pscheduler-result-archiver && "
    "sed -n '1,110p' archiver/response/operations_controller.py"
)

import logging

import connexion
from typing import Dict
from typing import Tuple
from typing import Union

from sqlalchemy import text, select

from archiver.common.globals import get_globals

logger = logging.getLogger(__name__)
from archiver.db.database_manager import DatabaseManager
from archiver.openapi_server.models.get_health200_response import GetHealth200Response  # noqa: E501
from archiver.openapi_server.models.get_schema200_response import GetSchema200Response  # noqa: E501
from archiver.openapi_server.models.status401_unauthorized import Status401Unauthorized  # noqa: E501
from archiver.openapi_server import util
from archiver.response.cors_response import cors_500

DBM = DatabaseManager.from_config(config=get_globals().config)

# -------- helpers (minimal, no external deps) --------

def _db_health() -> tuple[str, str | None]:
    """
    Returns ("ok"|"down", error_message_or_None).
    """
    try:
        with DBM.SessionLocal() as s:
            s.execute(text("SELECT 

In [11]:
""" Verify that our health-endpoint fix still works"""
# stdout, stderr = shore.execute(
#     """sudo docker exec archiver sh -c \
# 'curl -s -i -H "Authorization: Bearer $ARCHIVER_BEARER_TOKEN" \
# http://localhost:3500/ps/health'"""
# )
# stdout, stderr = ship.execute(
#     """sudo docker exec archiver sh -c \
# 'curl -s -i -H "Authorization: Bearer $ARCHIVER_BEARER_TOKEN" \
# http://localhost:3500/ps/health'"""
# )

HTTP/1.1 200 OK
Content-Length: 57
Content-Type: application/json
Date: Tue, 15 Sep 2026 20:43:52 GMT
Server: waitress

{
  "db": "ok",
  "status": "ok",
  "version": "1.1.0"
}
HTTP/1.1 200 OK
Content-Length: 57
Content-Type: application/json
Date: Tue, 15 Sep 2026 20:43:52 GMT
Server: waitress

{
  "db": "ok",
  "status": "ok",
  "version": "1.1.0"
}


In [14]:
 """ Dockers health check"""
# stdout, stderr = shore.execute(
#     "sudo docker inspect --format='{{.State.Health.Status}}' archiver"
# )

# stdout, stderr = ship.execute(
#     "sudo docker inspect --format='{{.State.Health.Status}}' archiver"
# )

healthy
healthy


### 4c. Cross-Connectivity Test — Ship ↔ Shore

Verify that the ship and shore VMs can reach each other's archiver endpoints over the FabNetv4 network, and that the ship's testpoint can reach the shore for pScheduler tests.

In [ ]:
# ------------------------------------------------------------
# Step 12: Validate ship-to-shore connectivity
# ------------------------------------------------------------
# Verify that the ship can reach the shore endpoint through
# both independent FabNetv4 networks before running pScheduler.
#
# Successful connectivity on both paths confirms that the
# experiment topology is ready for active measurements.

In [43]:
# ------------------------------------------------------------
# Pre-Experiment Recovery and Validation
# ------------------------------------------------------------
# Refresh FABRIC objects and re-apply saved node configuration.
# This allows the notebook to be reopened later without
# rebuilding the slice.

slice = fablib.get_slice(name=slice_name)

ship = slice.get_node(name=ship_name)
shore = slice.get_node(name=shore_name)

ship.config(refresh=True)
shore.config(refresh=True)

print("FABRIC node configuration refreshed.")

FABRIC node configuration refreshed.


In [44]:
# ------------------------------------------------------------
# Validate data-plane interfaces before running perfSONAR
# ------------------------------------------------------------

print("=== SHIP ===")
stdout, stderr = ship.execute("""
ip -br addr
echo "--- routes ---"
ip route
""")
print(stdout)

print("\n=== SHORE ===")
stdout, stderr = shore.execute("""
ip -br addr
echo "--- routes ---"
ip route
""")
print(stdout)

=== SHIP ===
lo               UNKNOWN        127.0.0.1/8 ::1/128 
enp3s0           UP             10.30.6.171/23 metric 100 2605:2800:2011:201:f816:3eff:fed2:64c2/64 fe80::f816:3eff:fed2:64c2/64 
br-0a0d53e3d45a  UP             172.18.0.1/16 fe80::7816:10ff:fea0:e430/64 
docker0          DOWN           172.17.0.1/16 fd6f:9352:6046::1/64 
veth9826fc3@if2  UP             fe80::8430:5fff:fed9:ca07/64 
veth4eb51ed@if2  UP             fe80::703d:dcff:fe04:9ca8/64 
veth31bdb04@if2  UP             fe80::18cc:50ff:fec7:4506/64 
veth19e2d75@if2  UP             fe80::42b:60ff:fea6:e808/64 
vethb55a073@if2  UP             fe80::98c8:3bff:feda:bec2/64 
--- routes ---
10.30.6.0/23 dev enp3s0 proto kernel scope link src 10.30.6.171 metric 100 
10.30.6.11 dev enp3s0 proto dhcp scope link src 10.30.6.171 metric 100 
169.254.169.254 via 10.30.6.11 dev enp3s0 proto dhcp src 10.30.6.171 metric 100 
172.17.0.0/16 dev docker0 proto kernel scope link src 172.17.0.1 linkdown 
172.18.0.0/16 dev br-0a0d53e3d45

In [45]:
# ------------------------------------------------------------
# Check FABRIC reservation state for ship data NICs
# ------------------------------------------------------------

slice = fablib.get_slice(name=slice_name)
ship = slice.get_node(name=ship_name)

print("=== SHIP COMPONENTS IN FABRIC ===")
ship.list_components()

print("\n=== SHIP INTERFACES IN FABRIC ===")
ship.list_interfaces()

=== SHIP COMPONENTS IN FABRIC ===


Name,Short Name,Details,Disk,Units,PCI Address,Model,Type,Device,Node,Numa Node
ship-testpoint-net2_IPv4_TACC_nic,net2_IPv4_TACC_nic,Mellanox ConnectX-6 VPI MCX653 dual port 100Gbps,0,1,0000:e2:06.4,NIC_Basic,SharedNIC,None,ship-testpoint,4
ship-testpoint-net1_IPv4_TACC_nic,net1_IPv4_TACC_nic,Mellanox ConnectX-6 VPI MCX653 dual port 100Gbps,0,1,0000:e2:08.0,NIC_Basic,SharedNIC,None,ship-testpoint,4



=== SHIP INTERFACES IN FABRIC ===


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
ship-testpoint-net2_IPv4_TACC_nic-p1,p1,ship-testpoint,net2_IPv4_TACC,100,,1E:89:33:CB:74:70,,enp8s0,auto,10.130.150.2,4,HundredGigE0/0/0/11
ship-testpoint-net1_IPv4_TACC_nic-p1,p1,ship-testpoint,net1_IPv4_TACC,100,,1E:84:21:A5:BD:99,,enp7s0,auto,10.130.151.2,4,HundredGigE0/0/0/11


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
ship-testpoint-net2_IPv4_TACC_nic-p1,p1,ship-testpoint,net2_IPv4_TACC,100,,1E:89:33:CB:74:70,,enp8s0,auto,10.130.150.2,4,HundredGigE0/0/0/11
ship-testpoint-net1_IPv4_TACC_nic-p1,p1,ship-testpoint,net1_IPv4_TACC,100,,1E:84:21:A5:BD:99,,enp7s0,auto,10.130.151.2,4,HundredGigE0/0/0/11


In [46]:
# ------------------------------------------------------------
# Recover missing FABRIC NICs after reboot
# ------------------------------------------------------------
# FABRIC still shows both NICs attached to the ship node,
# but Linux is not currently enumerating them.
# Ask the guest OS to rescan its PCI bus.

stdout, stderr = ship.execute("""
echo "=== BEFORE RESCAN ==="
lspci | grep -i -E 'ethernet|network'

sudo sh -c 'echo 1 > /sys/bus/pci/rescan'

sleep 3

echo
echo "=== AFTER RESCAN ==="
lspci | grep -i -E 'ethernet|network'

echo
echo "=== NETWORK INTERFACES ==="
ip -br addr
""")

print(stdout)
print(stderr)

=== BEFORE RESCAN ===
03:00.0 Ethernet controller: Red Hat, Inc. Virtio 1.0 network device (rev 01)

=== AFTER RESCAN ===
03:00.0 Ethernet controller: Red Hat, Inc. Virtio 1.0 network device (rev 01)

=== NETWORK INTERFACES ===
lo               UNKNOWN        127.0.0.1/8 ::1/128 
enp3s0           UP             10.30.6.171/23 metric 100 2605:2800:2011:201:f816:3eff:fed2:64c2/64 fe80::f816:3eff:fed2:64c2/64 
br-0a0d53e3d45a  UP             172.18.0.1/16 fe80::7816:10ff:fea0:e430/64 
docker0          DOWN           172.17.0.1/16 fd6f:9352:6046::1/64 
veth9826fc3@if2  UP             fe80::8430:5fff:fed9:ca07/64 
veth4eb51ed@if2  UP             fe80::703d:dcff:fe04:9ca8/64 
veth31bdb04@if2  UP             fe80::18cc:50ff:fec7:4506/64 
veth19e2d75@if2  UP             fe80::42b:60ff:fea6:e808/64 
vethb55a073@if2  UP             fe80::98c8:3bff:feda:bec2/64 
=== BEFORE RESCAN ===
03:00.0 Ethernet controller: Red Hat, Inc. Virtio 1.0 network device (rev 01)

=== AFTER RESCAN ===
03:00.0 Ethern

In [47]:
# ------------------------------------------------------------
# Final pre-repair inventory
# ------------------------------------------------------------

slice = fablib.get_slice(name=slice_name)

print("=== NODES ===")
slice.list_nodes()

print("\n=== NETWORKS ===")
slice.list_networks()

print("\n=== INTERFACES ===")
slice.list_interfaces()

=== NODES ===


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
999a7c1a-2da3-4f82-a4f3-cdfbb4e57516,ship-testpoint,4,16,100,default_ubuntu_24,qcow2,tacc-w4.fabric-testbed.net,TACC,ubuntu,2605:2800:2011:201:f816:3eff:fed2:64c2,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fed2:64c2,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
e2f5c1e4-5b17-41f0-ae35-bace8c729bc5,shore-testpoint,8,16,100,default_ubuntu_24,qcow2,ncsa-w2.fabric-testbed.net,NCSA,ubuntu,2620:0:c80:1001:f816:3eff:fe5b:7a0d,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:c80:1001:f816:3eff:fe5b:7a0d,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key



=== NETWORKS ===


ID,Name,Layer,Type,Site,Gateway,Subnet,State,Error
c2ab30d7-6694-4982-8b74-ea812ea21e2f,FABNET_IPv4_NCSA,L3,FABNetv4,NCSA,10.132.137.1,10.132.137.0/24,Active,
4c8905bd-992e-48ca-97c7-7232fbf9531c,net1_IPv4_TACC,L3,FABNetv4,TACC,10.130.151.1,10.130.151.0/24,Active,
7fbedb5a-b291-4333-9f91-1a4f05674c81,net2_IPv4_TACC,L3,FABNetv4,TACC,10.130.150.1,10.130.150.0/24,Active,



=== INTERFACES ===


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
shore-testpoint-FABNET_IPv4_NCSA_nic-p1,p1,shore-testpoint,FABNET_IPv4_NCSA,100,,12:53:E3:FD:D0:2E,enp7s0,enp7s0,auto,10.132.137.2,4,HundredGigE0/0/0/9
ship-testpoint-net2_IPv4_TACC_nic-p1,p1,ship-testpoint,net2_IPv4_TACC,100,,1E:89:33:CB:74:70,,enp8s0,auto,10.130.150.2,4,HundredGigE0/0/0/11
ship-testpoint-net1_IPv4_TACC_nic-p1,p1,ship-testpoint,net1_IPv4_TACC,100,,1E:84:21:A5:BD:99,,enp7s0,auto,10.130.151.2,4,HundredGigE0/0/0/11


Name,Short Name,Node,Network,Bandwidth,VLAN,MAC,Physical Device,Device,Mode,IP Address,Numa Node,Switch Port
shore-testpoint-FABNET_IPv4_NCSA_nic-p1,p1,shore-testpoint,FABNET_IPv4_NCSA,100,,12:53:E3:FD:D0:2E,enp7s0,enp7s0,auto,10.132.137.2,4,HundredGigE0/0/0/9
ship-testpoint-net2_IPv4_TACC_nic-p1,p1,ship-testpoint,net2_IPv4_TACC,100,,1E:89:33:CB:74:70,,enp8s0,auto,10.130.150.2,4,HundredGigE0/0/0/11
ship-testpoint-net1_IPv4_TACC_nic-p1,p1,ship-testpoint,net1_IPv4_TACC,100,,1E:84:21:A5:BD:99,,enp7s0,auto,10.130.151.2,4,HundredGigE0/0/0/11


In [48]:
# ------------------------------------------------------------
# Inspect available interface/network methods before repair
# ------------------------------------------------------------

slice = fablib.get_slice(name=slice_name)

net1 = slice.get_network(name="net1_IPv4_TACC")
net2 = slice.get_network(name="net2_IPv4_TACC")

iface1 = ship.get_interface(network_name="net1_IPv4_TACC")
iface2 = ship.get_interface(network_name="net2_IPv4_TACC")

print("=== NET1 METHODS ===")
print([m for m in dir(net1) if "interface" in m.lower()])

print("\n=== NET2 METHODS ===")
print([m for m in dir(net2) if "interface" in m.lower()])

print("\n=== IFACE1 METHODS ===")
print([m for m in dir(iface1) if "delete" in m.lower() or "network" in m.lower()])

print("\n=== IFACE2 METHODS ===")
print([m for m in dir(iface2) if "delete" in m.lower() or "network" in m.lower()])

=== NET1 METHODS ===
['_interfaces_cache', 'add_interface', 'get_interface', 'get_interfaces', 'has_interface', 'interfaces', 'remove_interface']

=== NET2 METHODS ===
['_interfaces_cache', 'add_interface', 'get_interface', 'get_interfaces', 'has_interface', 'interfaces', 'remove_interface']

=== IFACE1 METHODS ===
['delete', 'get_network', 'network', 'set_network']

=== IFACE2 METHODS ===
['delete', 'get_network', 'network', 'set_network']


In [49]:
# ------------------------------------------------------------
# Check component-management methods before NIC repair
# ------------------------------------------------------------

print("=== SHIP COMPONENT METHODS ===")
print([
    m for m in dir(ship)
    if "component" in m.lower()
])

print("\n=== CURRENT SHIP COMPONENT NAMES ===")
for component in ship.get_components():
    print(
        component.get_name(),
        "| model:", component.get_model()
    )

=== SHIP COMPONENT METHODS ===
['add_component', 'components', 'get_component', 'get_components', 'list_components']

=== CURRENT SHIP COMPONENT NAMES ===
ship-testpoint-net2_IPv4_TACC_nic | model: NIC_Basic
ship-testpoint-net1_IPv4_TACC_nic | model: NIC_Basic


### 5. Execute the Experiment

* Initiate network tests from the **ship-side Testpoint** targeting the **shore-side endpoint**.
* Test results are automatically collected and archived on both ends — **locally on the ship** and **remotely on the shore** — ensuring redundancy and continuous data availability.

In [22]:
# Verify the testpoint container is running and cron is set up (entrypoint handles this automatically)
stdout, stderror = ship.execute("sudo docker exec perfsonar-testpoint crontab -l")

no crontab for root


In [34]:
# ------------------------------------------------------------
# Step 13: Execute the perfSONAR experiment
# ------------------------------------------------------------
# Run active network measurements from the ship testpoint
# toward the shore endpoint through both FabNetv4 paths.
#
# Measurements performed:
#   1. Clock offset
#   2. One-way latency
#   3. Path MTU
#   4. Round-trip time (RTT)
#   5. Throughput
#   6. Traceroute
#
# The --reverse option also runs applicable tests in the
# reverse direction.
#
# Results are archived:
#   - locally on the ship
#   - remotely on the shore
#
# This provides redundant result storage for later analysis.
stdout, stderror = ship.execute(f'sudo docker exec perfsonar-testpoint /usr/bin/python3 /usr/src/app/periodic.py --hosts {hosts} --output-dir /data --archiver-urls {archive_urls} --auth-token {ARCHIVER_TOKEN} --reverse')

INFO: Logging to /data/logs/pscheduler_run_20260915-211620Z.log
INFO: Auth via ARCHIVER_BEARER or ARCHIVER_API_KEY (overridden by --auth-token / AUTH_TOKEN).
INFO: Archiver endpoints: https://localhost:8443/ps, https://10.132.137.2:8443/ps
INFO: Auth picked from env if set: AUTH_TOKEN
INFO: Hosts: 10.132.137.2@shore-testpoint-via-net1%enp7s0, 10.132.137.2@shore-testpoint-via-net2%enp8s0
INFO: Tests: clock, latency, mtu, rtt, throughput, trace
INFO: Tool mode: auto
INFO: Resolved interface 'enp7s0' -> 10.130.151.2
INFO: Host 10.132.137.2@shore-testpoint-via-net1%enp7s0: source binding -> 10.130.151.2
INFO: Running clock (auto) -> dest=10.132.137.2 name=shore-testpoint-via-net1 source=10.130.151.2 reverse=False
INFO: Running pscheduler task --format json --output /data/clock/shore-testpoint-via-net1@10.132.137.2_auto_20260915-211620Z_forward.json clock --dest 10.132.137.2 --source 10.130.151.2
INFO: Completed clock (auto) to 10.132.137.2 (shore-testpoint-via-net1), output: /data/clock/sh

In [120]:
stdout, stderr = ship.execute("echo SHIP_OK")
print(stdout)
print(stderr)

SHIP_OK
SHIP_OK




In [119]:
# ============================================================
# Safe check: is automatic perfSONAR scheduling configured?
# ============================================================

stdout, stderr = ship.execute("""
echo "=== CRON FILES ==="
sudo timeout 10 docker exec perfsonar-testpoint \
    ls -lah /etc/cron.d

echo
echo "=== CRON CONTENTS ==="
sudo timeout 10 docker exec perfsonar-testpoint \
    sh -c 'for f in /etc/cron.d/*; do
              echo "--- $f ---";
              cat "$f" 2>/dev/null;
           done'

echo
echo "=== RECENT PERFSONAR LOGS ==="
sudo timeout 10 docker exec perfsonar-testpoint \
    sh -c 'ls -lt /data/logs 2>/dev/null | head -15'
""")

print(stdout)

if stderr:
    print("STDERR:")
    print(stderr)

=== CRON FILES ===
total 28K
drwxr-xr-x 1 root root 4.0K Oct 18 05:48 .
drwxr-xr-x 1 root root 4.0K Oct 18 05:48 ..
-rw-r--r-- 1 root root  102 Mar 23  2022 .placeholder
-rw-r--r-- 1 root root  201 Jan  8  2022 e2scrub_all
-rw-r--r-- 1 root root  372 Oct 18 05:48 pscheduler_cron
-rw-r--r-- 1 root root  396 Feb  2  2021 sysstat

=== CRON CONTENTS ===
--- /etc/cron.d/e2scrub_all ---
30 3 * * 0 root test -e /run/systemd/system || SERVICE_MODE=1 /usr/lib/x86_64-linux-gnu/e2fsprogs/e2scrub_all_cron
10 3 * * * root test -e /run/systemd/system || SERVICE_MODE=1 /sbin/e2scrub_all -A -r
--- /etc/cron.d/pscheduler_cron ---
0 */6 * * * root /usr/bin/python3 /usr/src/app/periodic.py --hosts 10.132.137.2@shore-testpoint-via-net1\%enp8s0,10.132.137.2@shore-testpoint-via-net2\%enp7s0 --output-dir /data --reverse --archiver-urls https://localhost:8443/ps,https://10.132.137.2:8443/ps --auth-token ca98ad17840edd7b70257c227bcb7cbeb9cd5619279112e43d96e140aafd9033 >> /data/pscheduler_cron.log 2>&1

--- /et

In [122]:
stdout, stderr = ship.execute(r"""
sudo docker exec perfsonar-testpoint bash -c '
echo "=== CURRENT PSCHEDULER CRON ==="
sed -E "s/(--auth-token )[A-Za-z0-9]+/\1[HIDDEN]/" \
    /etc/cron.d/pscheduler_cron
'
""")

print(stdout)

if stderr:
    print(stderr)

=== CURRENT PSCHEDULER CRON ===
0 */6 * * * root /usr/bin/python3 /usr/src/app/periodic.py --hosts 10.132.137.2@shore-testpoint-via-net1\%enp8s0 10.132.137.2@shore-testpoint-via-net2\%enp7s0 --output-dir /data --reverse --archiver-urls https://localhost:8443/ps,https://10.132.137.2:8443/ps --auth-token [HIDDEN] >> /data/pscheduler_cron.log 2>&1

=== CURRENT PSCHEDULER CRON ===
0 */6 * * * root /usr/bin/python3 /usr/src/app/periodic.py --hosts 10.132.137.2@shore-testpoint-via-net1\%enp8s0 10.132.137.2@shore-testpoint-via-net2\%enp7s0 --output-dir /data --reverse --archiver-urls https://localhost:8443/ps,https://10.132.137.2:8443/ps --auth-token [HIDDEN] >> /data/pscheduler_cron.log 2>&1




### 6. **Monitoring and Visualization**

Users can monitor and analyze network performance using **Grafana dashboards** available on **both VMs**.

* **Access Grafana (Shore or Ship)** – via `https://127.0.0.1:8443` after establishing an SSH tunnel.
* The dashboards provide visualizations for:

  * Latency and RTT trends
  * Throughput and bandwidth tests
  * Path MTU discovery
  * Clock offset measurements
  * Traceroute visualizations

**Initial Grafana Setup:**

1. Log in with the default credentials `admin/perfsonar` — you'll be prompted to change the password on first login.
2. In the left navigation pane, go to **Data Sources → perfsonar-postgres**.
3. Under **Authentication**, set the password to `change_me`, then click **Save & Test**.

   * A message should confirm **"Database Connection OK."**
   * If you see a message indicating the password is being updated by another entity, refresh the page and click **Save & Test** again.
4. To explore the visualizations, navigate to **Dashboards → pScheduler → pscheduler-result-archiver (Pairs)**.

#### Create SSH Tunnel Configuration

* Use the generated SSH command to tunnel access to:

  * **Grafana Dashboard** (Result Archiver): [https://127.0.0.1:8443](https://127.0.0.1)


In [19]:
# ------------------------------------------------------------
# Step 14: Access Grafana visualization
# ------------------------------------------------------------
# Grafana reads the archived perfSONAR measurement results
# and displays them as dashboards.
#
# Grafana does not execute the network tests itself in this
# notebook; the measurements are produced by perfSONAR/pScheduler.
fablib.create_ssh_tunnel_config(overwrite=True)


SSH tunnel config created and zipped at: /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz

Download Instructions:
Download your custom `fabric_ssh_tunnel_tools.tgz` file from the `fabric_config` folder.

Usage Instructions:
1. Unzip the archive and place the resulting `fabric_ssh_tunnel_tools/` folder somewhere accessible from your terminal.
2. Open a terminal window (on Windows, use PowerShell).
3. Use `cd` to navigate into the `fabric_ssh_tunnel_tools` folder.
4. In your terminal, run the SSH tunnel command generated by the next notebook cell.
    


In [36]:
# Port on your local machine that you want to map the File Browser to.
local_port=8443
# Local interface to map the File Browser to (can be `localhost`)
local_host='127.0.0.1'

# Port on the node used by the File Browser Service
target_port='8443'

for n in slice.get_nodes():
    # Username/node on FABRIC
    target_host=f'{n.get_username()}@{n.get_management_ip()}'
    print(f'Tunnel command for {n.get_name()}')
    print(f'ssh  -L {local_host}:{local_port}:127.0.0.1:{target_port} -i {os.path.basename(fablib.get_default_slice_public_key_file())[:-4]} -F ssh_config {target_host}')
    print(f"View grafana https://{local_host}:{local_port}")

    local_port += 1
    print()


Tunnel command for shore-testpoint
ssh  -L 127.0.0.1:8443:127.0.0.1:8443 -i slice_key -F ssh_config ubuntu@2620:0:c80:1001:f816:3eff:fe5b:7a0d
View grafana https://127.0.0.1:8443

Tunnel command for ship-testpoint
ssh  -L 127.0.0.1:8444:127.0.0.1:8443 -i slice_key -F ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fed2:64c2
View grafana https://127.0.0.1:8444



In [115]:
# ============================================================
# Generate exact SSH tunnel commands for current slice nodes
# ============================================================

local_port = 8443
local_host = "127.0.0.1"
target_port = 8443

for n in slice.get_nodes():

    node_name = n.get_name()
    mgmt_ip = n.get_management_ip()

    # Use FABRIC's current SSH config/key paths
    ssh_key = os.path.basename(
        fablib.get_default_slice_public_key_file()
    )[:-4]

    print(f"Tunnel command for {node_name}")

    print(
        f"ssh -L 127.0.0.1:{local_port}:127.0.0.1:{target_port} "
        f"-i {ssh_key} "
        f"-F ssh_config "
        f"ubuntu@{mgmt_ip}"
    )

    print(
        f"View grafana https://{local_host}:{local_port}"
    )

    local_port += 1
    print()

Tunnel command for shore-testpoint
ssh -L 127.0.0.1:8443:127.0.0.1:8443 -i slice_key -F ssh_config ubuntu@2620:0:c80:1001:f816:3eff:fe5b:7a0d
View grafana https://127.0.0.1:8443

Tunnel command for ship-testpoint-new
ssh -L 127.0.0.1:8444:127.0.0.1:8443 -i slice_key -F ssh_config ubuntu@2605:2800:2011:201:f816:3eff:fe24:8569
View grafana https://127.0.0.1:8444



#### Example Grafana Display
![Display](./images/grafana.png)

In [17]:
print("=== SHORE CONTAINERS ===")

stdout, stderr = shore.execute("""
sudo docker ps \
--format 'table {{.Names}}\\t{{.Status}}\\t{{.Ports}}'
""")

print(stdout)
print(stderr)

=== SHORE CONTAINERS ===
NAMES                    STATUS                 PORTS
archiver                 Up 5 weeks (healthy)   3500/tcp
grafana                  Up 5 weeks             3000/tcp
grafana-image-renderer   Up 5 weeks (healthy)   8081/tcp
archiver-nginx           Up 4 weeks             80/tcp, 0.0.0.0:8443->443/tcp, [::]:8443->443/tcp
timescaledb              Up 5 weeks (healthy)   5432/tcp
NAMES                    STATUS                 PORTS
archiver                 Up 5 weeks (healthy)   3500/tcp
grafana                  Up 5 weeks             3000/tcp
grafana-image-renderer   Up 5 weeks (healthy)   8081/tcp
archiver-nginx           Up 4 weeks             80/tcp, 0.0.0.0:8443->443/tcp, [::]:8443->443/tcp
timescaledb              Up 5 weeks (healthy)   5432/tcp




In [18]:
print("=== SHORE ARCHIVER HEALTH ===")

stdout, stderr = shore.execute("""
sudo docker exec archiver sh -c \
'curl -s -H "Authorization: Bearer $ARCHIVER_BEARER_TOKEN" \
http://localhost:3500/ps/health'
""")

print(stdout)
print(stderr)

=== SHORE ARCHIVER HEALTH ===
{
  "db": "ok",
  "status": "ok",
  "version": "1.1.0"
}
{
  "db": "ok",
  "status": "ok",
  "version": "1.1.0"
}




In [ ]:
fablib.create_ssh_tunnel_config(overwrite=True)

In [111]:
# ============================================================
# Compact latest metrics for Net1 and Net2
# ============================================================

stdout, stderr = ship.execute(r"""
DB_USER=grafana_writer
DB_NAME=perfsonar

sudo docker exec timescaledb \
  psql -U "$DB_USER" -d "$DB_NAME" -P pager=off -c "
WITH ranked AS (
    SELECT
        ts,
        metric_name,
        test_type,
        dst,
        metric_value,
        unit,
        ROW_NUMBER() OVER (
            PARTITION BY dst, metric_name
            ORDER BY ts DESC
        ) AS rn
    FROM ps_test_results
    WHERE dst IN (
        'shore-testpoint-via-net1',
        'shore-testpoint-via-net2'
    )
      AND status = 'success'
)
SELECT
    dst,
    metric_name,
    ROUND(metric_value::numeric, 3) AS value,
    unit
FROM ranked
WHERE rn = 1
ORDER BY dst, metric_name;
"
""")

print(stdout)

if stderr:
    print(stderr)

           dst            |   metric_name   |   value    | unit  
--------------------------+-----------------+------------+-------
 shore-testpoint-via-net1 | avg_latency     |     13.850 | ms
 shore-testpoint-via-net1 | clock_diff_ms   |     75.904 | ms
 shore-testpoint-via-net1 | hop_count       |      4.000 | count
 shore-testpoint-via-net1 | loss_pct        |      0.000 | pct
 shore-testpoint-via-net1 | max_rtt_ms      |     31.706 | ms
 shore-testpoint-via-net1 | mean_rtt_ms     |     29.794 | ms
 shore-testpoint-via-net1 | min_rtt_ms      |     27.818 | ms
 shore-testpoint-via-net1 | mtu_bytes       |   9000.000 | bytes
 shore-testpoint-via-net1 | retransmits     | 286249.000 | count
 shore-testpoint-via-net1 | throughput_mbps |   8557.312 | mbps
 shore-testpoint-via-net2 | avg_latency     |     14.182 | ms
 shore-testpoint-via-net2 | clock_diff_ms   |     74.198 | ms
 shore-testpoint-via-net2 | hop_count       |      4.000 | count
 shore-testpoint-via-net2 | loss_pct        |  

# Automated Archive Analysis Extensions
The following cells extend the original perfSONAR-on-FABRIC experiment with four reproducible analysis functions: full-archive path comparison, a project-defined quality score, automatic degraded-path detection, and an automated experiment summary. Successful outputs are retained so the results can be inspected without rerunning the FABRIC deployment.


In [13]:
# ============================================================
# TASK 1 - AUTOMATIC FULL-ARCHIVE PATH COMPARISON
#
# Reads ALL valid archived measurements directly from
# TimescaleDB every time this cell is executed.
#
# No metric values are hard-coded.
# ============================================================

import csv
import io


# ------------------------------------------------------------
# 1. Query the complete valid archive
# ------------------------------------------------------------

query = r"""
WITH archive_summary AS (

    SELECT
        CASE
            WHEN dst = 'shore-testpoint-via-net1' THEN 'Net1'
            WHEN dst = 'shore-testpoint-via-net2' THEN 'Net2'
        END AS path,

        metric_name,
        COUNT(*) AS samples,
        AVG(metric_value) AS avg_value,
        MIN(metric_value) AS min_value,
        MAX(metric_value) AS max_value,
        MAX(unit) AS unit

    FROM ps_test_results

    WHERE dst IN (
        'shore-testpoint-via-net1',
        'shore-testpoint-via-net2'
    )

      AND status = 'success'

      AND metric_name IN (
          'throughput_mbps',
          'avg_latency',
          'mean_rtt_ms',
          'loss_pct',
          'retransmits',
          'hop_count',
          'mtu_bytes'
      )

    GROUP BY path, metric_name
)

SELECT
    path,
    metric_name,
    samples,
    avg_value,
    min_value,
    max_value,
    unit

FROM archive_summary

ORDER BY path, metric_name;
"""


stdout, stderr = shore.execute(
    f"""sudo docker exec timescaledb \
    psql -U grafana_writer -d perfsonar \
    -A -F ',' -t -c "{query}" """
)


# ------------------------------------------------------------
# 2. Convert database output into Python dictionaries
# ------------------------------------------------------------

metrics = {
    "Net1": {},
    "Net2": {}
}

for row in csv.reader(io.StringIO(stdout.strip())):

    if len(row) != 7:
        continue

    path, metric, samples, avg_value, min_value, max_value, unit = row

    metrics[path][metric] = {
        "samples": int(samples),
        "avg": float(avg_value),
        "min": float(min_value),
        "max": float(max_value),
        "unit": unit
    }


# ------------------------------------------------------------
# 3. Get archive date range automatically
# ------------------------------------------------------------

date_query = r"""
SELECT
    MIN(ts),
    MAX(ts)
FROM ps_test_results
WHERE dst IN (
    'shore-testpoint-via-net1',
    'shore-testpoint-via-net2'
)
AND status = 'success';
"""

date_stdout, date_stderr = shore.execute(
    f"""sudo docker exec timescaledb \
    psql -U grafana_writer -d perfsonar \
    -A -F ',' -t -c "{date_query}" """
)

dates = date_stdout.strip().split(",")

first_record = dates[0][:10]
latest_record = dates[1][:10]


# ------------------------------------------------------------
# 4. Comparison rules
# ------------------------------------------------------------

comparison_rules = {
    "throughput_mbps": ("Throughput (Mbps)", "higher"),
    "avg_latency": ("Latency (ms)", "lower"),
    "mean_rtt_ms": ("Mean RTT (ms)", "lower"),
    "loss_pct": ("Packet Loss (%)", "lower"),
    "retransmits": ("Retransmits", "lower"),
    "hop_count": ("Hop Count", "lower")
}


def choose_better(metric, rule):

    net1_value = metrics["Net1"][metric]["avg"]
    net2_value = metrics["Net2"][metric]["avg"]

    if net1_value == net2_value:
        return "Tie"

    if rule == "higher":
        return "Net1" if net1_value > net2_value else "Net2"

    if rule == "lower":
        return "Net1" if net1_value < net2_value else "Net2"


# ------------------------------------------------------------
# 5. Print automatic comparison
# ------------------------------------------------------------

print()
print("=" * 70)
print("           FULL ARCHIVE AUTOMATIC PATH COMPARISON")
print("=" * 70)
print()

for metric, (label, rule) in comparison_rules.items():

    n1 = metrics["Net1"][metric]["avg"]
    n2 = metrics["Net2"][metric]["avg"]

    winner = choose_better(metric, rule)

    print(
        f"{label:<20} "
        f"Net1={n1:>10.3f}   "
        f"Net2={n2:>10.3f}   "
        f"Better: {winner}"
    )


# MTU is mainly checked for consistency/equality
n1_mtu = metrics["Net1"]["mtu_bytes"]["avg"]
n2_mtu = metrics["Net2"]["mtu_bytes"]["avg"]

mtu_result = "Tie" if n1_mtu == n2_mtu else "Different"

print(
    f"{'MTU (bytes)':<20} "
    f"Net1={n1_mtu:>10.0f}   "
    f"Net2={n2_mtu:>10.0f}   "
    f"Result: {mtu_result}"
)


# ------------------------------------------------------------
# 6. Archive information
# ------------------------------------------------------------

print()
print("Archive records analyzed:")

for path in ["Net1", "Net2"]:

    normal_samples = metrics[path]["avg_latency"]["samples"]
    throughput_samples = metrics[path]["throughput_mbps"]["samples"]

    print(
        f"{path}: {normal_samples} measurement cycles, "
        f"{throughput_samples} throughput records"
    )

print(f"Archive period: {first_record} to {latest_record}")

print()
print("Task 1: Automatic comparison of archived network paths completed.")

Net1,avg_latency,95,13.821599999999984,13.789999999999997,14.129999999999999,ms
Net1,hop_count,95,4,4,4,count
Net1,loss_pct,95,0,0,0,pct
Net1,mean_rtt_ms,95,27.680842105263153,27.592,29.794,ms
Net1,mtu_bytes,95,9000,9000,9000,bytes
Net1,retransmits,190,94212.61578947368,36542,392535,count
Net1,throughput_mbps,190,9387.482359486245,5189.656227638622,9615.636147765885,mbps
Net2,avg_latency,95,13.82461052631578,13.780000000000001,14.181999999999999,ms
Net2,hop_count,95,4.031578947368421,4,6,count
Net2,loss_pct,95,0,0,0,pct
Net2,mean_rtt_ms,95,27.67895789473685,27.583,28.701,ms
Net2,mtu_bytes,95,9000,9000,9000,bytes
Net2,retransmits,190,92890.75263157894,42127,340460,count
Net2,throughput_mbps,190,9410.006380334868,8218.793278490202,9573.190931199591,mbps
2026-09-12 00:00:16.697296+00,2026-10-19 12:09:22.487972+00

           FULL ARCHIVE AUTOMATIC PATH COMPARISON

Throughput (Mbps)    Net1=  9387.482   Net2=  9410.006   Better: Net2
Latency (ms)         Net1=    13.822   Net2=    13.825  

In [16]:
# ============================================================
# TASK 2 - AUTOMATIC PATH QUALITY SCORE
#
# Uses full historical archive averages.
# No metric values are hard-coded.
#
# Score weights:
# Throughput   = 40%
# Mean RTT     = 30%
# Packet Loss  = 20%
# Latency      = 10%
# ============================================================

import csv
import io


# ------------------------------------------------------------
# 1. Read full archive averages directly from TimescaleDB
# ------------------------------------------------------------

query = r"""
SELECT
    CASE
        WHEN dst = 'shore-testpoint-via-net1' THEN 'Net1'
        WHEN dst = 'shore-testpoint-via-net2' THEN 'Net2'
    END AS path,

    metric_name,

    AVG(metric_value) AS avg_value

FROM ps_test_results

WHERE dst IN (
    'shore-testpoint-via-net1',
    'shore-testpoint-via-net2'
)

AND status = 'success'

AND metric_name IN (
    'throughput_mbps',
    'avg_latency',
    'mean_rtt_ms',
    'loss_pct'
)

GROUP BY path, metric_name

ORDER BY path, metric_name;
"""


stdout, stderr = shore.execute(
    f"""sudo docker exec timescaledb \
    psql -U grafana_writer -d perfsonar \
    -A -F ',' -t -c "{query}" """
)


# ------------------------------------------------------------
# 2. Store averages in Python
# ------------------------------------------------------------

metrics = {
    "Net1": {},
    "Net2": {}
}

for row in csv.reader(io.StringIO(stdout.strip())):

    if len(row) != 3:
        continue

    path, metric, value = row

    metrics[path][metric] = float(value)


# ------------------------------------------------------------
# 3. Find reference values for normalization
# ------------------------------------------------------------

max_throughput = max(
    metrics["Net1"]["throughput_mbps"],
    metrics["Net2"]["throughput_mbps"]
)

min_rtt = min(
    metrics["Net1"]["mean_rtt_ms"],
    metrics["Net2"]["mean_rtt_ms"]
)

min_latency = min(
    metrics["Net1"]["avg_latency"],
    metrics["Net2"]["avg_latency"]
)


# ------------------------------------------------------------
# 4. Calculate quality score
# ------------------------------------------------------------

def calculate_quality_score(data):

    # Higher throughput is better
    throughput_score = (
        data["throughput_mbps"] / max_throughput
    ) * 100

    # Lower RTT is better
    rtt_score = (
        min_rtt / data["mean_rtt_ms"]
    ) * 100

    # Lower latency is better
    latency_score = (
        min_latency / data["avg_latency"]
    ) * 100

    # Lower packet loss is better
    # 0% loss = 100 points
    # 5% or more = 0 points
    loss_score = max(
        0,
        100 - (data["loss_pct"] * 20)
    )

    final_score = (
        throughput_score * 0.40 +
        rtt_score        * 0.30 +
        loss_score       * 0.20 +
        latency_score    * 0.10
    )

    return final_score


score_net1 = calculate_quality_score(metrics["Net1"])
score_net2 = calculate_quality_score(metrics["Net2"])


# ------------------------------------------------------------
# 5. Select recommended path
# ------------------------------------------------------------

if score_net1 > score_net2:
    recommended = "Net1"

elif score_net2 > score_net1:
    recommended = "Net2"

else:
    recommended = "Tie"


# ------------------------------------------------------------
# 6. Display result
# ------------------------------------------------------------

print()
print("=" * 65)
print("              AUTOMATIC PATH QUALITY SCORE")
print("=" * 65)

for path in ["Net1", "Net2"]:

    print()
    print(path)
    print("-" * 25)

    print(
        f"Throughput : "
        f"{metrics[path]['throughput_mbps']:.3f} Mbps"
    )

    print(
        f"Latency    : "
        f"{metrics[path]['avg_latency']:.3f} ms"
    )

    print(
        f"Mean RTT   : "
        f"{metrics[path]['mean_rtt_ms']:.3f} ms"
    )

    print(
        f"Packet Loss: "
        f"{metrics[path]['loss_pct']:.3f}%"
    )

    score = score_net1 if path == "Net1" else score_net2

    print(
        f"Quality Score: {score:.2f}/100"
    )


print()
print("=" * 65)
print(f"RECOMMENDED PATH: {recommended}")
print("=" * 65)

print()
print("Score weights:")
print("Throughput  = 40%")
print("Mean RTT    = 30%")
print("Packet Loss = 20%")
print("Latency     = 10%")

Net1,avg_latency,13.821599999999984
Net1,loss_pct,0
Net1,mean_rtt_ms,27.680842105263153
Net1,throughput_mbps,9387.482359486245
Net2,avg_latency,13.82461052631578
Net2,loss_pct,0
Net2,mean_rtt_ms,27.67895789473685
Net2,throughput_mbps,9410.006380334868

              AUTOMATIC PATH QUALITY SCORE

Net1
-------------------------
Throughput : 9387.482 Mbps
Latency    : 13.822 ms
Mean RTT   : 27.681 ms
Packet Loss: 0.000%
Quality Score: 99.90/100

Net2
-------------------------
Throughput : 9410.006 Mbps
Latency    : 13.825 ms
Mean RTT   : 27.679 ms
Packet Loss: 0.000%
Quality Score: 100.00/100

RECOMMENDED PATH: Net2

Score weights:
Throughput  = 40%
Mean RTT    = 30%
Packet Loss = 20%
Latency     = 10%


## Step 3 — Automatic Degraded-Path Detection

This step extends the archive analysis by automatically identifying measurements that deviate from each path's historical behavior.

The detector uses archive-derived thresholds rather than fixed metric values. A measurement is flagged when it shows unusually high RTT or latency, unusually low throughput, packet loss, unusually high retransmissions, or a hop-count change.

In [17]:
# ============================================================
# TASK 3 - AUTOMATIC DEGRADED-PATH DETECTION
#
# Reads historical measurements directly from TimescaleDB.
# Thresholds are derived independently for Net1 and Net2.
#
# Detection rules:
#   RTT          > mean + 2 standard deviations
#   Latency      > mean + 2 standard deviations
#   Throughput   < mean - 2 standard deviations
#   Retransmits  > mean + 2 standard deviations
#   Packet Loss  > 0%
#   Hop Count    != most common historical hop count
#
# This is a rule-based statistical detector, not an ML model.
# ============================================================

import csv
import io
import statistics
from collections import Counter, defaultdict


# ------------------------------------------------------------
# 1. Read individual historical measurements from archive
# ------------------------------------------------------------

query = r"""
SELECT
    CASE
        WHEN dst = 'shore-testpoint-via-net1' THEN 'Net1'
        WHEN dst = 'shore-testpoint-via-net2' THEN 'Net2'
    END AS path,

    ts,
    metric_name,
    metric_value

FROM ps_test_results

WHERE dst IN (
    'shore-testpoint-via-net1',
    'shore-testpoint-via-net2'
)

AND status = 'success'

AND metric_name IN (
    'throughput_mbps',
    'avg_latency',
    'mean_rtt_ms',
    'loss_pct',
    'retransmits',
    'hop_count'
)

ORDER BY ts, path, metric_name;
"""


stdout, stderr = shore.execute(
    f"""sudo docker exec timescaledb \
    psql -U grafana_writer -d perfsonar \
    -A -F ',' -t -c "{query}" """
)


# ------------------------------------------------------------
# 2. Organize measurements by path and metric
# ------------------------------------------------------------

records = {
    "Net1": defaultdict(list),
    "Net2": defaultdict(list)
}

for row in csv.reader(io.StringIO(stdout.strip())):

    if len(row) != 4:
        continue

    path, timestamp, metric, value = row

    if path not in records:
        continue

    records[path][metric].append({
        "timestamp": timestamp,
        "value": float(value)
    })


# ------------------------------------------------------------
# 3. Calculate archive-derived thresholds
# ------------------------------------------------------------

thresholds = {
    "Net1": {},
    "Net2": {}
}

for path in ["Net1", "Net2"]:

    for metric in [
        "throughput_mbps",
        "avg_latency",
        "mean_rtt_ms",
        "retransmits"
    ]:

        values = [
            item["value"]
            for item in records[path][metric]
        ]

        mean_value = statistics.mean(values)

        # Sample standard deviation
        std_value = (
            statistics.stdev(values)
            if len(values) > 1
            else 0
        )

        thresholds[path][metric] = {
            "mean": mean_value,
            "std": std_value
        }


    # Historical normal hop count = most common value
    hop_values = [
        item["value"]
        for item in records[path]["hop_count"]
    ]

    normal_hop_count = Counter(
        hop_values
    ).most_common(1)[0][0]

    thresholds[path]["hop_count"] = {
        "normal": normal_hop_count
    }


# ------------------------------------------------------------
# 4. Build automatic degradation detector
# ------------------------------------------------------------

degradation_events = {
    "Net1": [],
    "Net2": []
}


def add_event(
    path,
    timestamp,
    metric,
    value,
    threshold,
    reason
):

    degradation_events[path].append({
        "timestamp": timestamp,
        "metric": metric,
        "value": value,
        "threshold": threshold,
        "reason": reason
    })


for path in ["Net1", "Net2"]:

    # --------------------------------------------------------
    # RTT - unusually high
    # --------------------------------------------------------

    stats = thresholds[path]["mean_rtt_ms"]

    rtt_threshold = (
        stats["mean"]
        + 2 * stats["std"]
    )

    for item in records[path]["mean_rtt_ms"]:

        if item["value"] > rtt_threshold:

            add_event(
                path,
                item["timestamp"],
                "mean_rtt_ms",
                item["value"],
                rtt_threshold,
                "High RTT"
            )


    # --------------------------------------------------------
    # Latency - unusually high
    # --------------------------------------------------------

    stats = thresholds[path]["avg_latency"]

    latency_threshold = (
        stats["mean"]
        + 2 * stats["std"]
    )

    for item in records[path]["avg_latency"]:

        if item["value"] > latency_threshold:

            add_event(
                path,
                item["timestamp"],
                "avg_latency",
                item["value"],
                latency_threshold,
                "High latency"
            )


    # --------------------------------------------------------
    # Throughput - unusually low
    # --------------------------------------------------------

    stats = thresholds[path]["throughput_mbps"]

    throughput_threshold = (
        stats["mean"]
        - 2 * stats["std"]
    )

    for item in records[path]["throughput_mbps"]:

        if item["value"] < throughput_threshold:

            add_event(
                path,
                item["timestamp"],
                "throughput_mbps",
                item["value"],
                throughput_threshold,
                "Low throughput"
            )


    # --------------------------------------------------------
    # Retransmits - unusually high
    # --------------------------------------------------------

    stats = thresholds[path]["retransmits"]

    retransmit_threshold = (
        stats["mean"]
        + 2 * stats["std"]
    )

    for item in records[path]["retransmits"]:

        if item["value"] > retransmit_threshold:

            add_event(
                path,
                item["timestamp"],
                "retransmits",
                item["value"],
                retransmit_threshold,
                "High retransmits"
            )


    # --------------------------------------------------------
    # Packet loss - any non-zero value
    # --------------------------------------------------------

    for item in records[path]["loss_pct"]:

        if item["value"] > 0:

            add_event(
                path,
                item["timestamp"],
                "loss_pct",
                item["value"],
                0,
                "Packet loss detected"
            )


    # --------------------------------------------------------
    # Hop count - differs from historical mode
    # --------------------------------------------------------

    normal_hops = thresholds[path]["hop_count"]["normal"]

    for item in records[path]["hop_count"]:

        if item["value"] != normal_hops:

            add_event(
                path,
                item["timestamp"],
                "hop_count",
                item["value"],
                normal_hops,
                "Hop-count change"
            )


# ------------------------------------------------------------
# 5. Sort events chronologically
# ------------------------------------------------------------

for path in ["Net1", "Net2"]:

    degradation_events[path] = sorted(
        degradation_events[path],
        key=lambda x: x["timestamp"]
    )


# ------------------------------------------------------------
# 6. Display derived thresholds
# ------------------------------------------------------------

print()
print("=" * 72)
print("         ARCHIVE-DERIVED DEGRADATION THRESHOLDS")
print("=" * 72)

for path in ["Net1", "Net2"]:

    print()
    print(path)
    print("-" * 30)

    rtt = thresholds[path]["mean_rtt_ms"]
    latency = thresholds[path]["avg_latency"]
    throughput = thresholds[path]["throughput_mbps"]
    retrans = thresholds[path]["retransmits"]

    print(
        f"High RTT threshold       : "
        f"{rtt['mean'] + 2*rtt['std']:.3f} ms"
    )

    print(
        f"High latency threshold   : "
        f"{latency['mean'] + 2*latency['std']:.3f} ms"
    )

    print(
        f"Low throughput threshold : "
        f"{throughput['mean'] - 2*throughput['std']:.3f} Mbps"
    )

    print(
        f"High retransmit threshold: "
        f"{retrans['mean'] + 2*retrans['std']:.3f}"
    )

    print(
        f"Normal hop count         : "
        f"{thresholds[path]['hop_count']['normal']:.0f}"
    )

    print(
        "Packet-loss threshold    : "
        "> 0%"
    )


# ------------------------------------------------------------
# 7. Summarize detected degradation events
# ------------------------------------------------------------

print()
print("=" * 72)
print("             AUTOMATIC DEGRADED-PATH DETECTION")
print("=" * 72)

for path in ["Net1", "Net2"]:

    events = degradation_events[path]

    print()
    print(path)
    print("-" * 30)

    print(
        f"Total abnormal metric observations: "
        f"{len(events)}"
    )

    reason_counts = Counter(
        event["reason"]
        for event in events
    )

    if len(events) == 0:

        print("No degraded conditions detected.")

    else:

        for reason, count in sorted(
            reason_counts.items()
        ):

            print(
                f"{reason:<24}: {count}"
            )


# ------------------------------------------------------------
# 8. Display the first detected events
# ------------------------------------------------------------

print()
print("=" * 72)
print("             SAMPLE DETECTED EVENTS")
print("=" * 72)

for path in ["Net1", "Net2"]:

    print()
    print(path)
    print("-" * 30)

    events = degradation_events[path]

    if not events:

        print("No events detected.")
        continue

    for event in events[:10]:

        print(
            f"{event['timestamp'][:19]} | "
            f"{event['reason']:<22} | "
            f"value={event['value']:.3f} | "
            f"reference={event['threshold']:.3f}"
        )


print()
print(
    "Task 3: Automatic degraded-path detection completed."
)

Net1,2026-09-12 00:00:45.557588+00,avg_latency,13.819999999999999
Net1,2026-09-12 00:01:28.550846+00,loss_pct,0
Net1,2026-09-12 00:01:28.550846+00,mean_rtt_ms,27.697
Net1,2026-09-12 00:03:06.830036+00,retransmits,65019
Net1,2026-09-12 00:03:06.830036+00,throughput_mbps,9426.04387259456
Net1,2026-09-12 00:04:44.418231+00,retransmits,84213
Net1,2026-09-12 00:04:44.418231+00,throughput_mbps,9526.369658264619
Net1,2026-09-12 00:05:01.564525+00,hop_count,4
Net2,2026-09-12 00:05:34.541897+00,avg_latency,13.91
Net2,2026-09-12 00:06:07.621131+00,loss_pct,0
Net2,2026-09-12 00:06:07.621131+00,mean_rtt_ms,27.673
Net2,2026-09-12 00:10:05.285488+00,retransmits,59432
Net2,2026-09-12 00:10:05.285488+00,throughput_mbps,9419.2069330463
Net2,2026-09-12 00:11:41.348471+00,retransmits,84900
Net2,2026-09-12 00:11:41.348471+00,throughput_mbps,9512.157867195154
Net2,2026-09-12 00:11:58.540172+00,hop_count,4
Net1,2026-09-12 06:00:55.538987+00,avg_latency,13.809999999999999
Net1,2026-09-12 06:01:38.558541+00,l

## Step 4 — Automated Experiment Summary

This step combines the archive comparison, quality score, and automatic degradation detection into one concise experiment report.

Metric-level anomaly flags occurring within the same 15-minute period are grouped into a single degradation episode, so multiple abnormal metrics from one measurement cycle are not counted as separate incidents.

In [18]:
# ============================================================
# TASK 4 - AUTOMATED EXPERIMENT SUMMARY
#
# Combines results from Tasks 1-3:
#   Task 1 -> archive comparison
#   Task 2 -> path-quality score
#   Task 3 -> degraded-path detection
#
# Multiple abnormal metric observations occurring within
# 15 minutes are grouped into one degradation episode.
#
# No experiment-result values are hard-coded.
# ============================================================

from datetime import datetime
from collections import Counter


# ------------------------------------------------------------
# 1. Parse archive timestamps
# ------------------------------------------------------------

def parse_timestamp(ts):

    return datetime.fromisoformat(
        ts.replace("+00", "+00:00")
    )


# ------------------------------------------------------------
# 2. Group abnormal observations into degradation episodes
# ------------------------------------------------------------

def build_degradation_episodes(events, window_minutes=15):

    if not events:
        return []

    sorted_events = sorted(
        events,
        key=lambda x: parse_timestamp(x["timestamp"])
    )

    episodes = []

    current_episode = {
        "start": parse_timestamp(
            sorted_events[0]["timestamp"]
        ),
        "end": parse_timestamp(
            sorted_events[0]["timestamp"]
        ),
        "events": [sorted_events[0]]
    }

    for event in sorted_events[1:]:

        event_time = parse_timestamp(
            event["timestamp"]
        )

        gap_minutes = (
            event_time
            - current_episode["end"]
        ).total_seconds() / 60

        if gap_minutes <= window_minutes:

            current_episode["events"].append(event)
            current_episode["end"] = event_time

        else:

            episodes.append(current_episode)

            current_episode = {
                "start": event_time,
                "end": event_time,
                "events": [event]
            }

    episodes.append(current_episode)

    return episodes


episodes = {
    "Net1": build_degradation_episodes(
        degradation_events["Net1"]
    ),

    "Net2": build_degradation_episodes(
        degradation_events["Net2"]
    )
}


# ------------------------------------------------------------
# 3. Summarize each path
# ------------------------------------------------------------

def summarize_path(path):

    path_events = degradation_events[path]
    path_episodes = episodes[path]

    reason_counts = Counter(
        event["reason"]
        for event in path_events
    )

    quality_score = (
        score_net1
        if path == "Net1"
        else score_net2
    )

    # Use Step 3 archive records to get cycle count
    total_cycles = len(
        records[path]["avg_latency"]
    )

    degradation_rate = (
        len(path_episodes)
        / total_cycles * 100
        if total_cycles > 0
        else 0
    )

    # Project-defined descriptive rule
    if degradation_rate <= 5:
        stability_status = "STABLE"
    else:
        stability_status = "VARIABLE"

    return {
        "quality_score": quality_score,
        "total_cycles": total_cycles,
        "abnormal_observations": len(path_events),
        "degradation_episodes": len(path_episodes),
        "degradation_rate_pct": degradation_rate,
        "stability_status": stability_status,
        "reason_counts": reason_counts
    }


path_summary = {
    "Net1": summarize_path("Net1"),
    "Net2": summarize_path("Net2")
}


# ------------------------------------------------------------
# 4. Display concise automated report
# ------------------------------------------------------------

print()
print("=" * 76)
print("                  AUTOMATED EXPERIMENT SUMMARY")
print("=" * 76)

print()
print(f"Archive period : {first_record} to {latest_record}")
print("Paths analyzed : Net1 and Net2")
print()

for path in ["Net1", "Net2"]:

    summary = path_summary[path]

    print("-" * 76)
    print(path)
    print("-" * 76)

    print(
        f"Measurement cycles       : "
        f"{summary['total_cycles']}"
    )

    print(
        f"Quality score            : "
        f"{summary['quality_score']:.2f}/100"
    )

    print(
        f"Abnormal metric flags    : "
        f"{summary['abnormal_observations']}"
    )

    print(
        f"Degradation episodes     : "
        f"{summary['degradation_episodes']}"
    )

    print(
        f"Episode rate             : "
        f"{summary['degradation_rate_pct']:.2f}%"
    )

    print(
        f"Overall stability        : "
        f"{summary['stability_status']}"
    )

    if summary["reason_counts"]:

        print("Detected conditions:")

        for reason, count in sorted(
            summary["reason_counts"].items()
        ):

            print(
                f"  - {reason}: {count}"
            )

    else:

        print("Detected conditions      : None")

    print()


# ------------------------------------------------------------
# 5. Recommended path from Task 2
# ------------------------------------------------------------

print("=" * 76)
print("PATH RECOMMENDATION")
print("=" * 76)

print()
print(f"Recommended path: {recommended}")
print(f"Net1 quality score: {score_net1:.2f}/100")
print(f"Net2 quality score: {score_net2:.2f}/100")


# ------------------------------------------------------------
# 6. Compare stability between paths
# ------------------------------------------------------------

net1_episodes = path_summary["Net1"][
    "degradation_episodes"
]

net2_episodes = path_summary["Net2"][
    "degradation_episodes"
]

print()

if net1_episodes < net2_episodes:

    print(
        "Stability observation: "
        "Net1 experienced fewer detected degradation episodes."
    )

elif net2_episodes < net1_episodes:

    print(
        "Stability observation: "
        "Net2 experienced fewer detected degradation episodes."
    )

else:

    print(
        "Stability observation: "
        "Both paths experienced the same number of "
        "detected degradation episodes."
    )


# ------------------------------------------------------------
# 7. Show first and last detected degradation episode
# ------------------------------------------------------------

print()
print("=" * 76)
print("DEGRADATION TIMELINE")
print("=" * 76)

for path in ["Net1", "Net2"]:

    path_episodes = episodes[path]

    print()
    print(path)
    print("-" * 30)

    if not path_episodes:

        print("No degradation episodes detected.")
        continue

    first_episode = path_episodes[0]
    last_episode = path_episodes[-1]

    print(
        "First detected episode :",
        first_episode["start"].strftime(
            "%Y-%m-%d %H:%M:%S UTC"
        )
    )

    print(
        "Last detected episode  :",
        last_episode["end"].strftime(
            "%Y-%m-%d %H:%M:%S UTC"
        )
    )

    print(
        "Total episodes         :",
        len(path_episodes)
    )


# ------------------------------------------------------------
# 8. Final experiment interpretation
# ------------------------------------------------------------

print()
print("=" * 76)
print("OVERALL EXPERIMENT INTERPRETATION")
print("=" * 76)

print()

if recommended == "Net1":

    print(
        "Net1 has the higher archive-derived path-quality score."
    )

elif recommended == "Net2":

    print(
        "Net2 has the higher archive-derived path-quality score."
    )

else:

    print(
        "The two paths have equal archive-derived quality scores."
    )


if (
    path_summary["Net1"]["stability_status"] == "STABLE"
    and
    path_summary["Net2"]["stability_status"] == "STABLE"
):

    print(
        "Both paths are generally stable, with occasional "
        "archive-derived abnormal observations."
    )

else:

    print(
        "At least one path shows repeated variability according "
        "to the archive-derived detection rules."
    )


print()
print(
    "Note: quality scores, degradation thresholds, and stability "
    "labels are project-defined analytical rules and are not "
    "official perfSONAR metrics."
)

print()
print(
    "Task 4: Automated experiment summary completed."
)


                  AUTOMATED EXPERIMENT SUMMARY

Archive period : 2026-09-12 to 2026-10-19
Paths analyzed : Net1 and Net2

----------------------------------------------------------------------------
Net1
----------------------------------------------------------------------------
Measurement cycles       : 95
Quality score            : 99.90/100
Abnormal metric flags    : 22
Degradation episodes     : 17
Episode rate             : 17.89%
Overall stability        : VARIABLE
Detected conditions:
  - High RTT: 1
  - High latency: 1
  - High retransmits: 16
  - Low throughput: 4

----------------------------------------------------------------------------
Net2
----------------------------------------------------------------------------
Measurement cycles       : 95
Quality score            : 100.00/100
Abnormal metric flags    : 32
Degradation episodes     : 25
Episode rate             : 26.32%
Overall stability        : VARIABLE
Detected conditions:
  - High RTT: 2
  - High latency: 4
  